In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
project_dir = Path.cwd()
if not (project_dir / "data" / "credit_risk_dataset.csv").is_file():
    project_dir = project_dir.parent
df = pd.read_csv(project_dir / "data" / "credit_risk_dataset.csv")
artifact_dir = project_dir / "notebook"
artifact_dir.mkdir(exist_ok=True)

# EDA

In [ ]:
df.shape
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["loan_status"].unique()

# class Inbalance

In [ ]:
df["loan_status"].value_counts()

In [ ]:
sns.countplot(data=df,x="loan_status")
plt.show()


In [ ]:
numerical_cols = df.select_dtypes(include="number").columns



In [ ]:
for i in numerical_cols:
    plt.figure(figsize=(8,6))
    sns.boxplot(data=df,y=i)



In [ ]:
(df["person_age"] > 100).sum()

In [ ]:
(df["person_emp_length"] > 60).sum()

In [ ]:
df.describe()

# correlation check

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True)

# Data validation & Outliners Handling

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy.duplicated().sum()
df_copy.drop_duplicates(inplace=True)
df_copy.duplicated().sum()

In [ ]:

# Remove ages below 18 and above 100

df_copy = df_copy[df_copy["person_age"] >= 18 ]
df_copy = df_copy[df_copy["person_age"] < 100 ]

# Remove employment lenght greater than age or extreme outlienrs 

df_copy =df_copy[df_copy["person_emp_length"] <= df_copy["person_age"]]
df_copy =df_copy[df_copy["person_emp_length"] <= 60]


In [ ]:
# Remove the rows with zero or negative income and loan amount
df_copy = df_copy[df_copy["loan_amnt"] > 0]

In [ ]:
X = df_copy.drop("loan_status",axis=1)
y = df_copy["loan_status"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [ ]:

df.columns

In [ ]:
numerical_cols = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
]

categorical_cols = [
    'person_home_ownership',
    'loan_intent',
    'loan_grade',
    'cb_person_default_on_file'
]

# Handle Class inbalance 

In [ ]:
# class weigths
y_train.value_counts()

neg, pos = np.bincount(y_train)

pos
scale_weights = neg / pos
scale_weights

# Pipeline

**logestic Regression : inpute -> Scaler Numeric, impute -> One-hot-encoding**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [ ]:
preprocessor_lr = ColumnTransformer(
    transformers=[
        #Pipeline - for numerical cols
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scaler",StandardScaler())
        ]),numerical_cols),

        # Pipeline - for categorical cols
        
        ("cato",Pipeline([
            ("imputer",SimpleImputer(strategy="constant",fill_value="Missing")),
            ("onehot",OneHotEncoder(handle_unknown="ignore"))
        ]),categorical_cols)
    ]
)

In [ ]:
# XGBoost : Impute numerical only (no scaling) , impute -. one - hot encoding

preprocessor_Xg = ColumnTransformer(
    transformers=[
        #Pipeline - for numerical cols
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median"))
            
        ]),numerical_cols),

        # Pipeline - for categorical cols
        
        ("cato",Pipeline([
            ("imputer",SimpleImputer(strategy="constant",fill_value="Missing")),
            ("onehot",OneHotEncoder(handle_unknown="ignore"))
        ]),categorical_cols)
    ]
)

# cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring = {"roc_auc":"roc_auc","accuracy":"accuracy","precision":"precision","recall":"recall", "f1":"f1"}

In [ ]:
# logestic regression
from sklearn.linear_model import LogisticRegression
lr_cv_pipeline = Pipeline([
    ("preprocessor",preprocessor_lr),
    ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
])
lr_cv_pipeline

In [ ]:
from xgboost import XGBClassifier
XGB_cv_pipeline = Pipeline([
    ("preprocessor",preprocessor_Xg),
    ("classifier",XGBClassifier(
        n_estimators=300,max_depth=5,learning_rate=0.1,scale_pos_weight=scale_weights,random_state=42
    ))
])

In [ ]:
from sklearn.model_selection import cross_val_score,cross_validate

In [ ]:
for cv_name, pipe in [("logestic_regression",lr_cv_pipeline),
                      ("XGBoost",XGB_cv_pipeline)]:
    cv_result = cross_validate(pipe,X_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)

    print(cv_name)
    print(f"ROC-AUC : {cv_result["test_roc_auc"].mean()}")
    print(f"Accuracy : {cv_result["test_accuracy"].mean()}")
    print(f"precsion : {cv_result["test_precision"].mean()}")
    print(f"Recall : {cv_result["test_recall"].mean()}")
    print(f"f1 : {cv_result["test_f1"].mean()}")
    

    
    

# Evaluate Helper Function

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

import matplotlib.pyplot as plt
import seaborn as sns


def evaluate_model(model_name, model, X_test, y_test, threshold=None):

    

    # Probability of positive class (1)
    y_pred_prob = model.predict_proba(X_test)[:, 1]

    if threshold:
        y_pred = (y_pred_prob > threshold).astype(int)
    else:
        y_pred = model.predict(X_test)

    # -----------------------------
    # Evaluation Metrics
    # -----------------------------
    accuracy_s = accuracy_score(y_test, y_pred)
    precision_s = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
    recall_s = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
    f1_s = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_pred_prob
    )

    pr_auc = average_precision_score(
        y_test,
        y_pred_prob
    )

    # -----------------------------
    # Print Metrics
    # -----------------------------
    print("=" * 50)
    print(f"Model: {model_name}")
    print("=" * 50)

    print(f"Accuracy  : {accuracy_s:.4f}")
    print(f"Precision : {precision_s:.4f}")
    print(f"Recall    : {recall_s:.4f}")
    print(f"F1 Score  : {f1_s:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")

    # -----------------------------
    # Classification Report
    # -----------------------------
    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )

    # -----------------------------
    # Confusion Matrix
    # -----------------------------
    cm = confusion_matrix(
        y_test,
        y_pred
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cbar=False
    )

    plt.xlabel("Predicted Value")
    plt.ylabel("Actual Value")
    plt.title(f"{model_name} - Confusion Matrix")

    plt.show()

    # -----------------------------
    # Precision-Recall Curve
    # -----------------------------
    precisions, recalls, thresholds = precision_recall_curve(
        y_test,
        y_pred_prob
    )

    plt.figure(figsize=(7, 5))

    plt.plot(
        recalls,
        precisions
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(
        f"{model_name} - Precision-Recall Curve"
    )

    plt.grid(True)
    plt.show()

    # -----------------------------
    # Return Metrics
    # -----------------------------
    return {
        "Model": model_name,
        "Accuracy": accuracy_s,
        "Precision": precision_s,
        "Recall": recall_s,
        "F1": f1_s,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    }

# Baseline Model

In [ ]:
baseline_model = Pipeline([
    ("preprocessor",preprocessor_lr),
    ("classifier",LogisticRegression(class_weight="balanced", random_state=42))
])

In [ ]:
baseline_model.fit(X_train,y_train)

In [ ]:
evaluate_model("Baseline logestic Model",baseline_model, X_test,y_test)

*XGB *

In [ ]:
xg_model = Pipeline([
    ("preprocessor",preprocessor_Xg),
    ("classifier",XGBClassifier(scale_pos_weight=scale_weights,random_state=42))
])

In [ ]:
xg_model.fit(X_train,y_train)

In [ ]:
evaluate_model("XGBOOST Model",xg_model, X_test,y_test)

# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from scipy.stats import uniform, randint

In [ ]:
param_grid = {
    "classifier__n_estimators": randint(150,600),
    "classifier__max_depth":randint(3,9),
    "classifier__learning_rate":uniform(0.01,0.5),
    "classifier__subsample":uniform(0.6,0.4),
    "classifier__colsample_bytree":(0.6,0.4),
    "classifier__min_child_weight":randint(1,10),
    "classifier__gamma":uniform(0,5)
}

In [ ]:
random_search_xgboost = RandomizedSearchCV(xg_model,param_grid,n_iter=150,cv=cv,scoring="average_precision",n_jobs=-1,verbose=1)

In [ ]:
random_search_xgboost.fit(X_train,y_train)

In [ ]:
random_search_xgboost.best_params_

In [ ]:
random_search_xgboost.best_score_

In [ ]:
evaluate_model("XGBOOST Model",random_search_xgboost, X_test,y_test)

# threshold calculation

In [ ]:
# Get probability of class 1
prob = random_search_xgboost.predict_proba(X_test)[:, 1]
# Calculate precision and recall for different thresholds
precisions, recalls, thresholds = precision_recall_curve(
    y_test,
    prob
)

# Calculate F1 score for each threshold
f1_scores = (
    2 * (precisions[:-1] * recalls[:-1])
    / (precisions[:-1] + recalls[:-1] + 1e-9)
)

# Find threshold with highest F1
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]
best_model=random_search_xgboost.best_estimator_

print(f"Best Threshold : {best_threshold:.4f}")
print(f"Best F1 Score   : {best_f1:.4f}")


In [ ]:
best_model

# probability Calibaration

In [ ]:
# Calibrate XGboost Model

from sklearn.calibration import CalibratedClassifierCV,calibration_curve
calibrated_model = CalibratedClassifierCV(
    best_model,
    method="sigmoid",
    cv=cv
)
calibrated_model.fit(X_train,y_train)

In [ ]:
# Get porbabilites
uncalibrated = xg_model.predict_proba(X_test)[:,1]

In [130]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(estimator=xg_model, method="sigmoid", cv=5)

calibrated_model.fit(X_train, y_train)

calibrated = calibrated_model.predict_proba(X_test)[:, 1]
calibrated

array([0.58598845, 0.02742211, 0.9450173 , ..., 0.45344691, 0.02441969,
       0.01665709], shape=(6305,))

In [ ]:
# create calibration curves

actual_uncal,predicted_uncal = calibration_curve(y_test,uncalibrated,n_bins=10)

In [ ]:
# create calibration curves

actual_cal,predicted_cal = calibration_curve(y_test,calibrated,n_bins=10)

In [ ]:
# plot

plt.plot([0,1],[0,1],"k--",label="prefect")
plt.plot(
    predicted_uncal,
    actual_uncal,
    "o-",
    label="uncalibrated"
)

plt.plot(
    predicted_cal,
    actual_cal,
    "o-",
    label="calibrated"

)
plt.legend()

# SHAP

# SHAP is help to find out the contribution of each and every features or columns in model predictions

**Local EXPLANATION**


*It tell why model can predict only one perticular rows or student according to columns or features*


**Global Explanation**

*It tells which features or columns are generally most importtant for the model.*

**Local and Global Explanaiton is used in dimension reduction**

In [ ]:
import shap

print(shap.__version__)

In [ ]:
# get the Trained XGboost classifier model out of the pipeline.
XGBClassifier = xg_model.named_steps["classifier"]


In [ ]:
# Transform X_test using the sam epreprocessing used for training.
X_test_transformed = xg_model.named_steps["preprocessor"].transform(X_test)

In [ ]:
# Get feature names after one-hot encoding
feature_names = xg_model.named_steps["preprocessor"].get_feature_names_out()

feature_names

In [ ]:
# convert to a datafram for cleaner shap plot
X_test_df = pd.DataFrame(X_test_transformed,columns=feature_names)
X_test_df

In [ ]:
# create SHAP explainer build for tree_based model
explainer = shap.TreeExplainer(XGBClassifier)

In [ ]:
# calculate SHAP values for every row in the test set
shap_values = explainer.shap_values(X_test_df)
shap_values

In [ ]:
# Global Explanation

In [ ]:
shap.summary_plot(shap_values,X_test_df)

*Local Explainaiton*

In [ ]:
# Pick one row to explain , eg the 5th applicant in the test set
row_index = 5

In [ ]:
import shap
import matplotlib.pyplot as plt

# Select the row you want to explain
row_index = 5

# Create a SHAP explanation for the selected row
waterfall_explanation = shap.Explanation(
    values=shap_values[row_index],
    base_values=explainer.expected_value,
    data=X_test_df.iloc[row_index].values,
    feature_names=X_test_df.columns.tolist()
)

# Draw the waterfall plot
shap.plots.waterfall(waterfall_explanation)

plt.show()

# analysis the False positives & false negeatives

In [ ]:
# get prediction for the best preforming XGBoost on the test set

y_pred = random_search_xgboost.predict(X_test)


In [ ]:
# create datafram to easilty compare the acutral and predicted valeus

result_df = X_test.copy()
result_df["actual"] = y_test
result_df["predicted"] = y_pred


In [ ]:
result_df

In [ ]:
# Identify false Positive: Model predicted 1 but actual was 0.
false_positive = result_df[(result_df["actual"]==0) & (result_df["predicted"]==1)]
# Identify false Negative: Model predicted 0 but actual was 1.
false_negative = result_df[(result_df["actual"]==1) & (result_df["predicted"]==0)]

In [ ]:
print(f"False positive: {len(false_positive)}")
print(f"False Negative: {len(false_negative)}")

In [ ]:
# false negative >= false postive

In [ ]:
import joblib

joblib.dump(calibrated_model, artifact_dir / "credit_risk_model.pkl")

In [133]:
import numpy as np
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(
    y_test,
    calibrated
)

# Calculate F1 scores aligned with thresholds
f1_scores = (
    2 * precisions[:-1] * recalls[:-1]
    / (precisions[:-1] + recalls[:-1] + 1e-9)
)

# Find the threshold with the highest F1 score
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]

print("Best Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

Best Threshold: 0.6880030031921567
Best F1 Score: 0.8428511875041743


In [129]:
joblib.dump(best_threshold, artifact_dir / "best_threshold.pkl")

['best_threshold.pkl']